In [1]:
!pip install transformers[torch] datasets lyricsgenius pronouncing

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 16.9 MB/s eta 0:00:00
  Created wheel for pronouncing: filename=pronouncing-0.2.0-py2.py3-none-any.whl size=6234 sha256=4a74e34cf18b29060c4c2a41c3e57fa366c3594565b2ad013c061adbd324200e
  Stored in directory: /root/.cache/pip/wheels/a0/76/15/dfdf38731993cdc4e86fd6d949c70c0e9786cf00073d8114d4
Successfully built pronouncing


In [18]:
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline
)
from datasets import load_dataset

# 1. LOAD TOKENIZER AND MODEL
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 doesn't have a pad token by default; we add one for batch processing
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

model = GPT2LMHeadModel.from_pretrained(model_name).to("cuda")

# CRITICAL: Resize the model's embeddings to accommodate the new [PAD] token
model.resize_token_embeddings(len(tokenizer))

# 2. DATA PREPARATION (FIXED FOR RUNTIME ERROR)
text_file_path = "drake_lyrics.txt"

# Load the raw text
with open(text_file_path, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f.readlines() if len(line.strip()) > 0]

# Join everything into one giant string with the end-of-text token
full_text = f" {tokenizer.eos_token} ".join(lines)

# Tokenize the entire block at once
tokenized_full_text = tokenizer(full_text, truncation=False)["input_ids"]

# Chunk the data into fixed-size blocks of 128 tokens
block_size = 128
chunks = [tokenized_full_text[i : i + block_size] for i in range(0, len(tokenized_full_text), block_size)]

# Remove the last chunk if it's shorter than block_size (to keep tensors consistent)
if len(chunks[-1]) < block_size:
    chunks.pop()

# Create a proper dataset format for the Trainer
class RapDataset(torch.utils.data.Dataset):
    def __init__(self, chunks):
        self.chunks = chunks
    def __len__(self):
        return len(self.chunks)
    def __getitem__(self, i):
        return {"input_ids": torch.tensor(self.chunks[i]), "labels": torch.tensor(self.chunks[i])}

dataset = RapDataset(chunks)

# 3. TRAINING CONFIGURATION
training_args = TrainingArguments(
    output_dir="./drake_ai_model",
    # overwrite_output_dir=True,      # This works in current versions if placed correctly
    num_train_epochs=5,
    per_device_train_batch_size=4,
    save_steps=200,
    logging_steps=50,
    warmup_steps=100,
    fp16=True,                      # Uses GPU acceleration
    report_to="none"                # Skips WandB login prompts
)

# 4. START TRAINING
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

print("Starting training... This usually takes 5-10 minutes on a T4 GPU.")
trainer.train()

# 5. INFERENCE (TESTING)
print("\n--- Testing the Model ---")
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=0)

# Drake-style prompt
test_prompt = "I'm just saying, I could do better"
results = generator(test_prompt, max_length=100, temperature=0.8, top_k=50)
print(results[0]['generated_text'])

# 6. (OPTIONAL) GRADIO UI
# Run this to get a shareable link for your portfolio
!pip install -q gradio
import gradio as gr

def generate_lyrics(seed_text):
    output = generator(seed_text, max_length=80, temperature=0.85)[0]['generated_text']
    return output

demo = gr.Interface(
    fn=generate_lyrics,
    inputs="text",
    outputs="text",
    title="Drake Lyric AI Generator",
    description="Enter a line to start a Drake-style verse."
)
demo.launch(share=True)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Token indices sequence length is longer than the specified maximum sequence length for this model (243262 > 1024). Running this sequence through the model will result in indexing errors


Starting training... This usually takes 5-10 minutes on a T4 GPU.


Step,Training Loss
50,4.348914
100,3.377995
150,3.154624
200,3.198503
250,3.039865
300,3.134416
350,3.026257
400,3.067750
450,3.017943
500,2.883589


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'top_k', 'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Testing the Model ---
I'm just saying, I could do better 
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab6b5545fea74c779f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
!pip install -q transformers[torch] datasets torch gradio

In [21]:
import torch
import os
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline
)

# --- STEP 1: LOAD TOKENIZER AND MODEL ---
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 needs a PAD token for batching
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

model = GPT2LMHeadModel.from_pretrained(model_name).to("cuda")
model.resize_token_embeddings(len(tokenizer))

# --- STEP 2: DATA PREPARATION (CHUNKED) ---
text_file_path = "drake_lyrics.txt"

with open(text_file_path, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f.readlines() if len(line.strip()) > 0]

# Join everything with EOS token
full_text = f" {tokenizer.eos_token} ".join(lines)

# Ignore the 'Token indices sequence length' warning; we are chunking manually below
tokenized_full_text = tokenizer(full_text, truncation=False)["input_ids"]

# Chunk into blocks of 128
block_size = 128
chunks = [tokenized_full_text[i : i + block_size] for i in range(0, len(tokenized_full_text), block_size)]
if len(chunks[-1]) < block_size: chunks.pop()

class RapDataset(torch.utils.data.Dataset):
    def __init__(self, chunks):
        self.chunks = chunks
    def __len__(self):
        return len(self.chunks)
    def __getitem__(self, i):
        item = torch.tensor(self.chunks[i])
        return {"input_ids": item, "labels": item}

dataset = RapDataset(chunks)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# --- STEP 3: TRAINING CONFIGURATION (FIXED VERSION) ---
training_args = TrainingArguments(
    output_dir="./drake_ai_final",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    save_steps=500,
    logging_steps=50,
    warmup_steps=100,
    fp16=True,
    report_to="none"                # No WandB login required
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

print("🚀 Starting training... This will bypass the previous error.")
trainer.train()

# --- STEP 4: ADVANCED INFERENCE ---
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=0)

def generate_drake_verse(seed_text):
    output = generator(
        seed_text,
        max_new_tokens=100,
        do_sample=True,
        top_k=50,
        top_p=0.92,
        temperature=0.85,
        repetition_penalty=1.2
    )
    clean_text = output[0]['generated_text']
    return clean_text.replace('[PAD]', '').replace('<|endoftext|>', '').strip()

# --- STEP 5: GRADIO UI ---
import gradio as gr

demo = gr.Interface(
    fn=generate_drake_verse,
    inputs=gr.Textbox(lines=2, label="Your Opening Bar"),
    outputs=gr.Textbox(lines=12, label="AI Generated Drake Verse"),
    title="Certified Ghostwriter AI"
)

demo.launch(share=True)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Token indices sequence length is longer than the specified maximum sequence length for this model (243262 > 1024). Running this sequence through the model will result in indexing errors


🚀 Starting training... This will bypass the previous error.


Step,Training Loss
50,4.348916
100,3.377869
150,3.154835
200,3.198629
250,3.039663
300,3.134214
350,3.026065
400,3.067190
450,3.017508
500,2.881832


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://23be7b3bbb3ea12660.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [22]:
# 1. Create requirements.txt
with open("requirements.txt", "w") as f:
    f.write("transformers\ntorch\ndatasets\ngradio\n")

# 2. Create a basic README.md
readme_content = """# Drake-AI: Lyrical Ghostwriter
A fine-tuned GPT-2 model trained on Drake's discography to generate multi-line hip-hop verses.

## Features
- Fine-tuned Causal Language Model (CLM)
- Nucleus Sampling for creative generation
- Gradio Web Interface

## How to use
1. Install requirements: `pip install -r requirements.txt`
2. Run the notebook or script.
"""
with open("README.md", "w") as f:
    f.write(readme_content)